In [ ]:
""" Run in the CMD terminal -
cd D:\mini_project\RAG - path to the folder  
python -m venv rag_env - Creat environment
rag_env\Scripts\activate - Activate environment
pip install pymupdf pandas tqdm jupyter - install required libraries
code . - open VS code in the same env
"""
#Verify environment inside notebook
import sys
print(sys.executable)

d:\mini_project\RAG\rag_env\Scripts\python.exe


<>:1: SyntaxWarning: invalid escape sequence '\m'
<>:1: SyntaxWarning: invalid escape sequence '\m'
C:\Users\poona\AppData\Local\Temp\ipykernel_1076\2571513158.py:1: SyntaxWarning: invalid escape sequence '\m'
  """ Run in the CMD terminal -


In [ ]:
# STEP 1 - DOWNLOADING DOCUMENTS
""" 
Choose a keyword from your area of interest, eg. single cell rna sequencing
Search it on Pubmed, filter "free full text" 
Save the PMID as pmids.txt in the same directory.

Install pubmed2pdf (in terminal) - pip install pubmed2pdf
"""
#Download pdfs using pmids 
import subprocess

command = '''
Get-Content pmids.txt | ForEach-Object { pubmed2pdf pdf --pmids $_ --out D:\\mini_project\\RAG\\papers }
'''

subprocess.run(["powershell", "-Command", command])

# Verify 
import os

files = os.listdir("papers")
print(f"Total files: {len(files)}")

for f in files[:10]:
    print(f)

Total files: 15
36526433.pdf
37120627.pdf
38528186.pdf
38570797.pdf
38702773.pdf
38839954.pdf
38898291.pdf
38962008.pdf
39346541.pdf
39367943.pdf


In [4]:
# STEP 2 - TEXT EXTRACTION 

""" 
Install required library - pip install pymupdf
"""
import fitz  # PyMuPDF
import os
import json
from tqdm import tqdm

pdf_folder = "papers"
data = []
failed_files = []
empty_files = []

os.makedirs("data", exist_ok=True)

for file in tqdm(os.listdir(pdf_folder)):
    if not file.endswith(".pdf"):
        continue

    path = os.path.join(pdf_folder, file)

    try:
        with fitz.open(path) as doc:   # ensures file closes properly
            text_pages = []

            for page in doc:
                page_text = page.get_text().strip()
                
                if page_text:  # skip empty pages
                    text_pages.append(page_text)

            full_text = " ".join(text_pages)

            if not full_text:
                empty_files.append(file)
                continue

            data.append({
                "doc_id": file,
                "text": full_text
            })

    except Exception as e:
        print(f"Error reading {file}: {e}")
        failed_files.append(file)

# Save JSON
with open("data/raw_text.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)

print("\nExtraction complete!")
print(f"Total PDFs processed: {len(data)}")
print(f"Empty files: {len(empty_files)}")
print(f"Failed files: {len(failed_files)}")

100%|██████████| 15/15 [00:02<00:00,  5.59it/s]


Extraction complete!
Total PDFs processed: 15
Empty files: 0
Failed files: 0


In [7]:
# STEP 3 - CHUNKING (WITH OVERLAPS)

import json

with open("data/raw_text.json", "r", encoding="utf-8") as f:
    data = json.load(f)

chunk_size = 300
overlap = 50
chunked_data = []

for doc in data:
    doc_id = doc["doc_id"]
    words = doc["text"].split()
    
    start = 0
    passage_id = 0
    
    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunk_text = " ".join(chunk_words).strip()
        
        # Skip very tiny chunks (optional but useful)
        if len(chunk_words) > 30:
            chunked_data.append({
                "doc_id": doc_id,
                "passage_id": passage_id,
                "text": chunk_text,
                "word_count": len(chunk_words) 
            })
        
        start += (chunk_size - overlap)
        passage_id += 1

# Save
with open("data/chunked_data.json", "w", encoding="utf-8") as f:
    json.dump(chunked_data, f, indent=2)

print("Chunking complete!")
print("Total chunks:", len(chunked_data))

chunked_data[:3]

Chunking complete!
Total chunks: 681


[{'doc_id': '36526433.pdf',
  'passage_id': 0,
  'text': 'Cross-species cell-type assignment from single-cell RNA-seq data by a heterogeneous graph neural network Xingyan Liu,1,2,5 Qunlun Shen,1,2,5 and Shihua Zhang1,2,3,4 1NCMIS, CEMS, RCSDS, Academy of Mathematics and Systems Science, Chinese Academy of Sciences, Beijing 100190, China; 2School of Mathematical Sciences, University of Chinese Academy of Sciences, Beijing 100049, China; 3Center for Excellence in Animal Evolution and Genetics, Chinese Academy of Sciences, Kunming 650223, China; 4Key Laboratory of Systems Health Science of Zhejiang Province, School of Life Science, Hangzhou Institute for Advanced Study, University of Chinese Academy of Sciences, Hangzhou 310024, China Cross-species comparative analyses of single-cell RNA sequencing (scRNA-seq) data allow us to explore, at single-cell res- olution, the origins of the cellular diversity and evolutionary mechanisms that shape cellular form and function. Cell-type assignment 

In [11]:
# STEP 4 - EMBEDDINGS (FINAL OPTIMIZED VERSION)

from transformers import BertTokenizer, BertModel
import torch
import torch.nn.functional as F
import json
from tqdm import tqdm

# Load chunked data
with open("data/chunked_data.json", "r", encoding="utf-8") as f:
    chunked_data = json.load(f)

# Load BioBERT
model_name = "dmis-lab/biobert-base-cased-v1.1"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name, use_safetensors=True)

# 🔥 GPU setup
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

# 🔥 Parameters
batch_size = 16   # reduce to 8 if GPU memory error
max_length = 256  # optimized for speed

texts = [item["text"] for item in chunked_data]
embeddings_list = []

# 🔥 Batched embedding
for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i+batch_size]
    
    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    )
    
    # Move inputs to GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # CLS pooling
    batch_embeddings = outputs.last_hidden_state[:, 0, :]
    
    # Normalize (for cosine similarity / FAISS IP)
    batch_embeddings = F.normalize(batch_embeddings, p=2, dim=1)
    
    embeddings_list.extend(batch_embeddings.cpu().tolist())  # move back to CPU

# Assign embeddings back
for item, emb in zip(chunked_data, embeddings_list):
    item["embedding"] = emb

# Save
with open("data/embedded_data.json", "w", encoding="utf-8") as f:
    json.dump(chunked_data, f, indent=2)

print("BioBERT embeddings created!")
print("Total embeddings:", len(embeddings_list))
print("Embedding dimension:", len(embeddings_list[0]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 43/43 [00:31<00:00,  1.37it/s]


BioBERT embeddings created!
Total embeddings: 681
Embedding dimension: 768


In [13]:
# STEP 5 - VECTOR DATABASE 
""" 
Install FAISS - pip install faiss-cpu
"""
import json
import numpy as np
import faiss

# Load embedded data
with open("data/embedded_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total chunks:", len(data))

# Extract embeddings
embeddings = [item["embedding"] for item in data]
embeddings = np.array(embeddings).astype("float32")

print("Shape:", embeddings.shape)

# Create FAISS index (IMPORTANT: use IP for cosine similarity)
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)   # Cosine similarity
index.add(embeddings) # Add embeddings

print("FAISS index created!")
print("Total vectors in index:", index.ntotal)

# Save index
faiss.write_index(index, "data/faiss_index.bin")
print("Index file saved!")

Total chunks: 681
Shape: (681, 768)
FAISS index created!
Total vectors in index: 681
Index file saved!


In [14]:
# STEP 6 - DEFINING QUERIES AND QUERY EMBEDDINGS

import torch
import torch.nn.functional as F
import faiss

# DEFINE QUERIES
queries = [
    "What is single cell RNA sequencing?",
    "Applications of scRNA-seq in cancer",
    "How does scRNA-seq help in tumor analysis?",
    "Limitations of single cell sequencing",
    "Techniques used in scRNA-seq",
    "Role of scRNA-seq in immunology",
    "How does scRNA-seq work?",
    "Advantages of single cell sequencing",
    "Data analysis methods in scRNA-seq",
    "Future of scRNA-seq technology"
]

# LOAD FAISS INDEX
index = faiss.read_index("data/faiss_index.bin")
print("FAISS index loaded!")
print("Total vectors:", index.ntotal)

# QUERY EMBEDDING FUNCTION
def embed_query(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )
    
    # Move to GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # CLS pooling
    embedding = outputs.last_hidden_state[:, 0, :]
    
    # Normalize
    embedding = F.normalize(embedding, p=2, dim=1)
    
    return embedding.cpu().numpy().astype("float32")

FAISS index loaded!
Total vectors: 681


In [15]:
# STEP 7 - RETRIEVAL

import numpy as np
import json
import faiss

# Load embedded data
with open("data/embedded_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Load FAISS index
index = faiss.read_index("data/faiss_index.bin")

k = 5

for query in queries:
    print("\n" + "="*80)
    print("QUERY:", query)
    print("="*80)
    
    query = query.strip()
    
    query_variants = [
        query,
        query.lower()
    ]
    
    all_results = []
    
    # Search for each query variant
    for q in query_variants:
        q_embedding = embed_query(q)   # already float32 (1,768)
        
        distances, indices = index.search(q_embedding, k)
        
        for idx, dist in zip(indices[0], distances[0]):
            all_results.append((idx, dist))
    
    # Sort by similarity (important)
    all_results = sorted(all_results, key=lambda x: x[1], reverse=True)
    
    # Remove duplicates (keep best score)
    seen = set()
    unique_results = []
    
    for idx, dist in all_results:
        if idx not in seen:
            seen.add(idx)
            unique_results.append((idx, dist))
    
    # Take top k
    unique_results = unique_results[:k]
    
    # Display results
    for rank, (idx, dist) in enumerate(unique_results):
        result = data[idx]
        
        print(f"\nRank {rank+1}")
        print(f"Doc: {result['doc_id']}")
        print(f"Passage ID: {result['passage_id']}")
        print(f"Score: {dist:.4f}")   
        print("Text preview:")
        print(result["text"][:300])
        print("-"*50)


QUERY: What is single cell RNA sequencing?

Rank 1
Doc: 39346541.pdf
Passage ID: 20
Score: 0.8543
Text preview:
ECM deposition and organization, while macrophages efficiently degrade and modify it [37]. CellChat prediction suggests Nur77 may be involved in the pathogenesis of radiation-induced skin injury. Discussion The skin is particularly susceptible to radiation because it is a constantly renewing organ, 
--------------------------------------------------

Rank 2
Doc: 38898291.pdf
Passage ID: 28
Score: 0.8513
Text preview:
and the appropriate skills to perform these tasks. Engagement involves creating trust, allowing dynamic communication, and ultimately giving back to the community, through the discovery and increased awareness of the neuroscience and neuro- pathology133. Healing happens with the potential for therap
--------------------------------------------------

Rank 3
Doc: 38839954.pdf
Passage ID: 15
Score: 0.8510
Text preview:
AFF4 transcription factor correlated with bur

In [41]:
#STEP 8 - Connect Retrieval --> LLM (Answer Generation)

""" 
Download and install Ollama in C drive "https://ollama.com/download/windows" 
Add to PATH (Environment Variables)
If can't find, in CMD - 
    look for the .exe file - where ollama - then add to PATH
    test version - ollama --version
    Pull Model - ollama pull gemma:2b 
In rag_env install Ollama python client - pip install ollama
"""
# STEP 8 - CONNECT RETRIEVAL → LLM (FINAL)

import ollama

def answer_query(query, k=5):
    
    # CLEAN QUERY
    query = query.strip()
    
    query_variants = [
        query,
        query.lower()
    ]
    
    all_results = []
    
    # RETRIEVAL (FAISS)
    for q in query_variants:
        q_embedding = embed_query(q)   
        
        distances, indices = index.search(q_embedding, k)
        
        for idx, dist in zip(indices[0], distances[0]):
            all_results.append((idx, dist))
    
    # Sort by similarity 
    all_results = sorted(all_results, key=lambda x: x[1], reverse=True)
    
    # Remove duplicates
    seen = set()
    unique_results = []
    
    for idx, dist in all_results:
        if idx not in seen:
            seen.add(idx)
            unique_results.append((idx, dist))
    
    # Take top k
    unique_results = unique_results[:k]
    
    # BUILD CONTEXT
    retrieved_chunks = []
    refs = set()
    
    for idx, _ in unique_results:
        retrieved_chunks.append(data[idx]["text"])
        refs.add(data[idx]["doc_id"])
    
    context = "\n\n".join(retrieved_chunks)
    
    # PROMPT
    prompt = f"""
You are a scientific assistant.

Answer using ONLY the provided context.

Rules:
- Start directly with the answer
- No conversational phrases
- Keep it concise and clear
- Structure the answer in bullet points if possible
- If not found, say: Not found in provided papers

Context:
{context}

Question:
{query}
"""
    
    # LLM CALL
    response = ollama.chat(
        model="gemma:2b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    answer = response["message"]["content"]
    
    # REFERENCES
    ref_text = "\n".join([f"(Ref: {r})" for r in refs])
    
    return answer + "\n\n" + ref_text


print("RAG pipeline ready!")

RAG pipeline ready!


In [40]:
# ASK A QUESTION
while True:
    query = input("\nAsk a question: ").strip()
    
    # blank input → exit
    if query == "":
        print("Stopping...")
        break
    
    result = answer_query(query)
    
    print("\n" + "="*80)
    print("ANSWER:\n")
    print(result)
    print("="*80)


ANSWER:

- Sure, here's the answer to the question:

Single cell rna sequencing is a technique that allows for the direct and comprehensive observation of transcription within single cells
- This technique utilizes single cells as biological samples and employs advanced technologies such as microfluidics and high-throughput sequencing to analyze the transcriptome of individual cells
- By performing single-cell rna sequencing, researchers can obtain a detailed understanding of the transcription dynamics within different cell types, uncover subtle differences among individual cells, and elucidate complex biological processes.

(Ref: 38839954.pdf)
(Ref: 36526433.pdf)
(Ref: 38898291.pdf)
(Ref: 39346541.pdf)
Stopping...


In [ ]:
# STEP 9 - STREAMLIT 
"""
pip install streamlit
code - app.py
streamlit run app.py
"""

'\npip install streamlit\ncode - app.py\n'